In [1]:
from six.moves import urllib
opener = urllib.request.build_opener()
opener.addheaders = [('User-agent', 'Mozilla/5.0')]
urllib.request.install_opener(opener)

In [2]:
import torch
from torch import nn
import torch.nn.functional as F
from torchvision import datasets, transforms

# Define a transform to normalize the data
transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((0.5,), (0.5,)),
                              ])
# Download and load the training data
trainset = datasets.MNIST('~/.pytorch/MNIST_data/', download=True, train=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

In [3]:
# Build a feed-forward network
model = nn.Sequential(nn.Linear(784, 128),
                      nn.ReLU(),
                      nn.Linear(128, 64),
                      nn.ReLU(),
                      nn.Linear(64, 10))

# Define the loss
criterion = nn.CrossEntropyLoss()

# Get our data
images, labels = next(iter(trainloader))
# Flatten images
images = images.view(images.shape[0], -1)

# Forward pass, get our logits
logits = model(images)
# Calculate the loss with the logits and the labels
loss = criterion(logits, labels)

print(loss)

tensor(2.3109, grad_fn=<NllLossBackward0>)


In [4]:
x = torch.randn(2,2, requires_grad=True)
print(x)

tensor([[ 0.6920,  2.2149],
        [ 0.3371, -0.0963]], requires_grad=True)


In [5]:
y = x**2
print(y)

tensor([[0.4788, 4.9059],
        [0.1136, 0.0093]], grad_fn=<PowBackward0>)


In [6]:
print(y.grad_fn)


In [7]:
z = y.mean()
print(z)

tensor(1.3769, grad_fn=<MeanBackward0>)


In [8]:
print(x.grad)

None


In [9]:
z.backward()
print(x.grad)
print(x/2)

tensor([[ 0.3460,  1.1075],
        [ 0.1685, -0.0481]])
tensor([[ 0.3460,  1.1075],
        [ 0.1685, -0.0481]], grad_fn=<DivBackward0>)


In [10]:
# Build a feed-forward network
model = nn.Sequential(nn.Linear(784, 128),
                      nn.ReLU(),
                      nn.Linear(128, 64),
                      nn.ReLU(),
                      nn.Linear(64, 10),
                      nn.LogSoftmax(dim=1))

criterion = nn.NLLLoss()
images, labels = next(iter(trainloader))
images = images.view(images.shape[0], -1)

logps = model(images)
loss = criterion(logps, labels)

In [11]:
print('Before backward pass: \n', model[0].weight.grad)

loss.backward()

print('After backward pass: \n', model[0].weight.grad)

Before backward pass: 
 None
After backward pass: 
 tensor([[-0.0039, -0.0039, -0.0039,  ..., -0.0039, -0.0039, -0.0039],
        [-0.0016, -0.0016, -0.0016,  ..., -0.0016, -0.0016, -0.0016],
        [-0.0002, -0.0002, -0.0002,  ..., -0.0002, -0.0002, -0.0002],
        ...,
        [ 0.0004,  0.0004,  0.0004,  ...,  0.0004,  0.0004,  0.0004],
        [-0.0006, -0.0006, -0.0006,  ..., -0.0006, -0.0006, -0.0006],
        [-0.0014, -0.0014, -0.0014,  ..., -0.0014, -0.0014, -0.0014]])


In [12]:
from torch import optim

# Optimizers require the parameters to optimize and a learning rate
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [13]:
print('Initial weights - ', model[0].weight)

images, labels = next(iter(trainloader))
images.resize_(64, 784)

# Clear the gradients, do this because gradients are accumulated
optimizer.zero_grad()

# Forward pass, then backward pass, then update weights
output = model(images)
loss = criterion(output, labels)
loss.backward()
print('Gradient -', model[0].weight.grad)

Initial weights -  Parameter containing:
tensor([[ 0.0121, -0.0053, -0.0147,  ...,  0.0085,  0.0321,  0.0296],
        [ 0.0188,  0.0339,  0.0216,  ..., -0.0057, -0.0043, -0.0202],
        [ 0.0242,  0.0199, -0.0196,  ..., -0.0102,  0.0210,  0.0226],
        ...,
        [ 0.0334,  0.0197, -0.0286,  ...,  0.0170,  0.0313,  0.0283],
        [-0.0163,  0.0229,  0.0154,  ...,  0.0184,  0.0353,  0.0040],
        [ 0.0096,  0.0161, -0.0206,  ...,  0.0220,  0.0275,  0.0215]],
       requires_grad=True)
Gradient - tensor([[-0.0022, -0.0022, -0.0022,  ..., -0.0022, -0.0022, -0.0022],
        [-0.0010, -0.0010, -0.0010,  ..., -0.0010, -0.0010, -0.0010],
        [ 0.0015,  0.0015,  0.0015,  ...,  0.0015,  0.0015,  0.0015],
        ...,
        [-0.0003, -0.0003, -0.0003,  ..., -0.0003, -0.0003, -0.0003],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0015,  0.0015,  0.0015,  ...,  0.0015,  0.0015,  0.0015]])


In [14]:
# Take an update step and few the new weights
optimizer.step()
print('Updated weights - ', model[0].weight)

Updated weights -  Parameter containing:
tensor([[ 0.0121, -0.0053, -0.0147,  ...,  0.0085,  0.0321,  0.0296],
        [ 0.0188,  0.0339,  0.0216,  ..., -0.0057, -0.0043, -0.0202],
        [ 0.0242,  0.0199, -0.0196,  ..., -0.0103,  0.0209,  0.0226],
        ...,
        [ 0.0334,  0.0197, -0.0285,  ...,  0.0170,  0.0313,  0.0283],
        [-0.0163,  0.0229,  0.0154,  ...,  0.0184,  0.0353,  0.0040],
        [ 0.0095,  0.0161, -0.0206,  ...,  0.0220,  0.0275,  0.0215]],
       requires_grad=True)


In [15]:
model = nn.Sequential(nn.Linear(784, 128),
                      nn.ReLU(),
                      nn.Linear(128, 64),
                      nn.ReLU(),
                      nn.Linear(64, 10),
                      nn.LogSoftmax(dim=1))

criterion = nn.NLLLoss()
optimizer = optim.SGD(model.parameters(), lr=0.003)

epochs = 5
for e in range(epochs):
    running_loss = 0
    for images, labels in trainloader:
        # Flatten MNIST images into a 784 long vector
        images = images.view(images.shape[0], -1)
    
        # TODO: Training pass
        optimizer.zero_grad()
        
        output = model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    else:
        print(f"Training loss: {running_loss/len(trainloader)}")

Training loss: 1.8621809408862962
Training loss: 0.8055113942892567
Training loss: 0.5136053734687346
Training loss: 0.42566156276126405
Training loss: 0.38423431426413784


In [16]:
%matplotlib inline
import helper

images, labels = next(iter(trainloader))

img = images[0].view(1, 784)
# Turn off gradients to speed up this part
with torch.no_grad():
    logps = model(img)

# Output of the network are log-probabilities, need to take exponential for probabilities
ps = torch.exp(logps)
helper.view_classify(img.view(1, 28, 28), ps)

ModuleNotFoundError: No module named 'grp'